In [ ]:
%pylab inline
import eucare as ec
import networkx as nx

plotting_kwargs = {
    'figsize': (10, 10),
    'render_faces': False,
    'render_vertices': False,
    'render_edges': True,
    'face_inset': 0,
    'line_width': 3,
}

In [ ]:
from functools import reduce

class GridMolecule():
    def __init__(self, graph, top, bottom, right, left):
        self.top = top
        self.bottom = bottom
        self.right = right
        self.left = left
        self.graph = graph
        
    def glue_on_right(self, right_molecule):
        self.graph.glue_graph_e2e(right_molecule.graph, self.right, right_molecule.left)
        self.graph.recompute_positions()
        self.right = right_molecule.right
        return self
    
    def glue_on_bottom(self, bottom_molecule):
        self.graph.glue_graph_e2e(bottom_molecule.graph, self.bottom, bottom_molecule.top)
        self.graph.recompute_positions()
        self.bottom = bottom_molecule.bottom
        return self
    
    def rotate90(self):
        # rotate 90 degrees anti-clockwise
        self.top, self.bottom, self.right, self.left = self.right, self.left, self.bottom, self.top
        return self
    
    
def make_molecule_grid(molecule_dict):
    """molecule_dict should be a dict mapping positions (tuple of int) to (distinct) molecules"""
    molecule_graph = nx.Graph()
    # find connections
    for pos, molecule in molecule_dict.items():
        right_pos = (pos[0]+1, pos[1])
        bottom_pos = (pos[0], pos[1]+1)
        right_molecule = molecule_dict.get(right_pos, None)
        if right_molecule is not None:
            molecule_graph.add_edge(molecule, right_molecule, to_glue=(molecule.right, right_molecule.left))
        bottom_molecule = molecule_dict.get(bottom_pos, None)
        if bottom_molecule is not None:
            molecule_graph.add_edge(molecule, bottom_molecule, to_glue=(molecule.bottom, bottom_molecule.top))
    
    root_molecule = next(iter(molecule_graph.nodes))
    result = root_molecule.graph
    for m1, m2 in nx.bfs_edges(molecule_graph, source=root_molecule):
        to_glue = molecule_graph[m1][m2]['to_glue']
        assert not any([e in result.halfedges for e in m2.graph.halfedges])
        result.glue_graph_e2e(m2.graph, *to_glue)
        result.recompute_positions()  # FIXME: this should not be necessary..
    return result

    
class DragonscaleMolecule(GridMolecule):
    def __init__(self, a=1):
        G = nx.Graph()
        G.add_edges_from([
            ((0, 0), (-(1+a), -(1+a))), # middle to lower corners
            ((0, 0), (1+a, -(1+a))),

            ((0, 0), (1, 1)), # middle to upper cornerspositions, molecules
            ((0, 0), (-1, 1)),
        ], **{ec.overlap.CREASE_ASSIGNMENT: ec.overlap.MOUNTAIN})
        G.add_path(
            ((-1, 0), (0, 0), (1, 0)),   # middle to left/right
            **{ec.overlap.CREASE_ASSIGNMENT: ec.overlap.VALLEY})
        G.add_cycle([(-1, 0), (-(1+a), -a), (-(1+a), -(1+a)), (-1, -(1+2*a)), 
                     (1, -(1+2*a)), ((1+a), -(1+a)), ((1+a), -a), (1, 0),
                     (1, 1), (-1, 1)], 
                    **{ec.overlap.CREASE_ASSIGNMENT: ec.overlap.VALLEY})
        G, v_lookup = ec.conversions.EHEG_from_nx(G, return_v_lookup=True)
        G.recompute_lengths_and_angles()
        
        super(DragonscaleMolecule, self).__init__(
            graph = G,
            top = v_lookup[(-1, 1)].get_outgoing_border(),
            bottom = v_lookup[(1, -(1+2*a))].get_outgoing_border(),
            right = v_lookup[(1, 1)].get_outgoing_border(),
            left = v_lookup[(-(1+a), -(1+a))].get_outgoing_border()
        )
        
        self.v_lookup = v_lookup
        self.a = a
        
        
class DragonscaleMolecule(GridMolecule):
    def __init__(self, a=1, b=1):
        G = nx.Graph()
        G.add_edges_from([
            ((0, 0), (-(1+a), -(1+a))), # middle to lower corners
            ((0, 0), (1+b, -(1+b))),

            ((0, 0), (1, 1)), # middle to upper corners
            ((0, 0), (-1, 1)),
        ], **{ec.overlap.CREASE_ASSIGNMENT: ec.overlap.MOUNTAIN})
        
        G.add_path(
            ((-1, 0), (0, 0), (1, 0)),   # middle to left/right
            **{ec.overlap.CREASE_ASSIGNMENT: ec.overlap.VALLEY})
        
        G.add_cycle([(-1, 0), (-(1+a), -a), (-(1+a), -(1+a)), (-1 + b - a, -(1+a+b)), 
                     (1 + b - a, -(1+a+b)), ((1+b), -(1+b)), ((1+b), -b), (1, 0),
                     (1, 1), (-1, 1)], 
                    **{ec.overlap.CREASE_ASSIGNMENT: ec.overlap.VALLEY})
        
        G, v_lookup = ec.conversions.EHEG_from_nx(G, return_v_lookup=True)
        G.recompute_lengths_and_angles()
        
        super(DragonscaleMolecule, self).__init__(
            graph = G,
            top = v_lookup[(-1, 1)].get_outgoing_border(),
            bottom = v_lookup[(1 + b - a, -(1+a+b))].get_outgoing_border(),
            right = v_lookup[(1, 1)].get_outgoing_border(),
            left = v_lookup[(-(1+a), -(1+a))].get_outgoing_border()
        )
        
        self.v_lookup = v_lookup
        self.a = a
        
class DragonscaleMoleculeB(GridMolecule):
    def __init__(self, a=1, b=1, c=0):
        G = nx.Graph()
        G.add_edges_from([
            ((0, 0), (-(1+a), -(1+a))), # lower middle to lower corners
            ((0, 0), (1+b, -(1+b))),
            
            ((0, 0), (0, c)), # lower middle to upper middle

            ((0, 0+c), (1, 1+c)), # upper middle to upper corners
            ((0, 0+c), (-1, 1+c)),
        ], **{ec.overlap.CREASE_ASSIGNMENT: ec.overlap.MOUNTAIN})
        
        G.add_edges_from([
            ((-1, c), (0, c)),  # left to upper middle
            ((0, 0), (1, 0)),   # lower middle to right
        ], **{ec.overlap.CREASE_ASSIGNMENT: ec.overlap.VALLEY})
        
        G.add_cycle([(-1, c), (-(1+a), -a+c), (-(1+a), -(1+a)), (-1 + b - a, -(1+a+b)), 
                     (1 + b - a, -(1+a+b)), ((1+b), -(1+b)), ((1+b), -b), (1, 0),
                     (1, 1+c), (-1, 1+c)], 
                    **{ec.overlap.CREASE_ASSIGNMENT: ec.overlap.VALLEY})
        
        G, v_lookup = ec.conversions.EHEG_from_nx(G, return_v_lookup=True)
        G.recompute_lengths_and_angles()
        
        super(DragonscaleMoleculeB, self).__init__(
            graph = G,
            top = v_lookup[(-1, 1+c)].get_outgoing_border(),
            bottom = v_lookup[(1 + b - a, -(1+a+b))].get_outgoing_border(),
            right = v_lookup[(1, 1+c)].get_outgoing_border(),
            left = v_lookup[(-(1+a), -(1+a))].get_outgoing_border()
        )
        
        self.v_lookup = v_lookup
        self.a = a
        
def hexagon_square_mask(a, b, c):
    b=b-1
    coords = np.mgrid[:a+b,:b+c]
    img = np.ones(coords[0].shape)
    img[coords[0] + coords[1]<b] = 0
    img[coords[0] + coords[1]>a+b+c-2] = 0
    img = img[::-1]
    return img

import numpy as np
n = 7
a, b, c = n, n, n + 2
reps = 0.5
f = (n-1)/reps
offset = 1.0
amp = 1.0
p = 1.0
phase = np.pi  #-np.pi / 4
c_phase = 0

molecule_dict = {(i, j): DragonscaleMolecule(
    a=(offset+amp+amp*np.cos(2*np.pi*(j-i-a)/f + phase))**p, 
    b=(offset+amp+amp*np.cos(2*np.pi*(j-a)/f ))**p,
    #c=(1.1+1*np.cos(2*np.pi*(j-a)/f + c_phase))**p,
) for i, j in np.argwhere(hexagon_square_mask(a, b, c))}

tess = make_molecule_grid(molecule_dict)

# molecule_grid = [[DragonscaleMolecule(
#     a=(offset+amp+amp*np.sin(2*np.pi*(j-i)/f))**p, 
#     b=(offset+amp+amp*np.sin(2*np.pi*i/f))**p
# )
#                   for j in range(n)] 
#                  for i in range(n)]

# rows = [reduce(lambda row, molecule: row.glue_on_right(molecule), row_list) for row_list in molecule_grid]

# # for row in rows:
# #     row.graph.show(**plotting_kwargs)
    
# tess = reduce(lambda stack, row: stack.glue_on_bottom(row), rows).graph
tess.normalize_positions()
tess.show(**plotting_kwargs)

from eucare.overlap import fold_complete
tess.recompute_lengths_and_angles()
result = fold_complete(tess)

result['CP'].show(**plotting_kwargs)
result['folded_view_top'].show(**plotting_kwargs)
result['folded_view_bottom'].show(**plotting_kwargs)

ec.overlap.save_results(result, f'nice_images/verrillO_{n}')

In [ ]:
renderer = ec.redering.SvgwriteRenderer()
renderer.render_graph('test.svg', result['CP'], height=20)

In [ ]:

from matplotlib.collections import LineCollection, PolyCollection
from eucare.plotting import set_equal_aspect
from eucare.overlap import fold_wireframe

def get_polys(faces):
    return [[v['pos'] for v in f.vertex_iter()] for f in faces]

#faces = list(SRG.faces)
faces = list(result['CP'].faces)
fold_wireframe(result['CP'])

fig = plt.figure(figsize=(15, 15))
ax = fig.add_subplot(1, 1, 1)

pc = PolyCollection(get_polys(faces), antialiased=True, color='k')
pc.set_alpha(0.1)

polys = ax.add_collection(pc)

ax.autoscale()
set_equal_aspect()
plt.draw()

In [ ]:
ec.overlap.save_results(result, path='Verrill_Dragonscale_0', render_settings=plotting_kwargs)